In [11]:
# =============================================================================
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
# =============================================================================
import warnings
warnings.filterwarnings('ignore')
import time
import os
# --- Library Imports ---
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import catboost as cb
import optuna
print("Libraries imported successfully.")
# --- Helper Function for Winkler Score ---
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)
# --- Global Constants ---
N_SPLITS = 5
RANDOM_STATE = 42
DATA_PATH = './'
N_OPTUNA_TRIALS = 30 # A strong number for a comprehensive search
COMPETITION_ALPHA = 0.1

# --- Load Raw Data ---
try:
    # We drop the low-variance columns they identified right away
    drop_cols=['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm','view_otherwater', 'view_other']
    df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
    df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)
    print("Raw data loaded successfully.")
except FileNotFoundError:
    print("ERROR: Could not find 'dataset.csv' or 'test.csv'.")
    exit()
# --- Prepare Target Variable ---
y_true = df_train['sale_price'].copy()

# Create the 'grade_for_stratify' variable right after loading the data.
grade_for_stratify = df_train['grade'].copy()
# The mean-error model works best when predicting the raw price directly
# So, we will NOT log-transform the target this time.
# df_train.drop('sale_price', axis=1, inplace=True) # We keep sale_price for FE
print("Setup complete.")


Libraries imported successfully.
Raw data loaded successfully.
Setup complete.


In [12]:
# Make sure to have these libraries installed
# pip install pandas numpy scikit-learn

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
import gc

# Define a random state for reproducibility
RANDOM_STATE = 42

def create_comprehensive_features(df_train, df_test):
    """
    Combines original and new advanced feature engineering steps into a single pipeline.
    """
    print("--- Starting Comprehensive Feature Engineering ---")

    # Store original indices and target variable
    train_ids = df_train.index
    test_ids = df_test.index
    y_train = df_train['sale_price'].copy() # Keep the target separate

    # Combine for consistent processing
    df_train_temp = df_train.drop(columns=['sale_price'])
    all_data = pd.concat([df_train_temp, df_test], axis=0, ignore_index=True)

    # --- Original Feature Engineering ---

    # A) Brute-Force Numerical Interactions
    print("Step 1: Creating brute-force numerical interaction features...")
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1', 'grade', 'year_built']
    # Ensure all columns exist and are numeric, fill missing with 0 for safety
    for col in NUMS:
        if col not in all_data.columns:
            all_data[col] = 0
        else:
            all_data[col] = pd.to_numeric(all_data[col], errors='coerce').fillna(0)
            
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] * all_data[NUMS[j]]

    # B) Date Features
    print("Step 2: Creating date features...")
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['sale_year'] = all_data['sale_date'].dt.year
    all_data['sale_month'] = all_data['sale_date'].dt.month
    all_data['sale_dayofyear'] = all_data['sale_date'].dt.dayofyear
    all_data['age_at_sale'] = all_data['sale_year'] - all_data['year_built']

    # C) TF-IDF Text Features
    print("Step 3: Creating TF-IDF features for text columns...")
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=128, binary=True)
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        
        tfidf_matrix = tfidf.fit_transform(all_data[col])
        tfidf_svd = svd.fit_transform(tfidf_matrix)
        
        tfidf_df = pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])
        all_data = pd.concat([all_data, tfidf_df], axis=1)

    # D) Log transform some interaction features
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in all_data.columns:
            all_data[c] = np.log1p(all_data[c].fillna(0))

    # --- New Feature Engineering Ideas ---

    # F) Group-By Aggregation Features
    print("Step 4: Creating group-by aggregation features...")
    group_cols = ['submarket', 'city', 'zoning']
    num_cols_for_agg = ['grade', 'sqft', 'imp_val', 'land_val', 'age_at_sale']

    for group_col in group_cols:
        for num_col in num_cols_for_agg:
            agg_stats = all_data.groupby(group_col)[num_col].agg(['mean', 'std', 'max', 'min']).reset_index()
            agg_stats.columns = [group_col] + [f'{group_col}_{num_col}_{stat}' for stat in ['mean', 'std', 'max', 'min']]
            all_data = pd.merge(all_data, agg_stats, on=group_col, how='left')
            all_data[f'{num_col}_minus_{group_col}_mean'] = all_data[num_col] - all_data[f'{group_col}_{num_col}_mean']

    # G) Ratio Features
    print("Step 5: Creating ratio features...")
    # Add a small epsilon to prevent division by zero
    epsilon = 1e-6 
    all_data['total_val'] = all_data['imp_val'] + all_data['land_val']
    all_data['imp_val_to_land_val_ratio'] = all_data['imp_val'] / (all_data['land_val'] + epsilon)
    all_data['land_val_ratio'] = all_data['land_val'] / (all_data['total_val'] + epsilon)
    all_data['sqft_to_lot_ratio'] = all_data['sqft'] / (all_data['sqft_lot'] + epsilon)
    all_data['was_renovated'] = (all_data['year_reno'] > 0).astype(int)
    all_data['reno_age_at_sale'] = np.where(all_data['was_renovated'] == 1, all_data['sale_year'] - all_data['year_reno'], -1)

    # H) Geospatial Clustering Features
    print("Step 6: Creating geospatial clustering features...")
    coords = all_data[['latitude', 'longitude']].copy()
    coords.fillna(coords.median(), inplace=True) # Simple imputation

    # KMeans is sensitive to feature scaling, but for lat/lon it's often okay without it.
    kmeans = KMeans(n_clusters=20, random_state=RANDOM_STATE, n_init=10) 
    all_data['location_cluster'] = kmeans.fit_predict(coords)
    
    # Calculate distance to each cluster center
    cluster_centers = kmeans.cluster_centers_
    for i in range(len(cluster_centers)):
        center = cluster_centers[i]
        all_data[f'dist_to_cluster_{i}'] = np.sqrt((coords['latitude'] - center[0])**2 + (coords['longitude'] - center[1])**2)

    # --- Final Cleanup ---
    print("Step 7: Finalizing feature set...")
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
    all_data = all_data.drop(columns=cols_to_drop)

    # One-hot encode the new cluster feature
    all_data = pd.get_dummies(all_data, columns=['location_cluster'], prefix='loc_cluster')
    
    # Final check for any remaining object columns to be safe (besides index)
    object_cols = all_data.select_dtypes(include='object').columns
    if len(object_cols) > 0:
        print(f"Warning: Found unexpected object columns: {object_cols}. Dropping them.")
        all_data = all_data.drop(columns=object_cols)
        
    all_data.fillna(0, inplace=True)

    # Separate back into train and test sets
    train_len = len(train_ids)
    X = all_data.iloc[:train_len].copy()
    X_test = all_data.iloc[train_len:].copy()
    
    # Restore original indices
    X.index = train_ids
    X_test.index = test_ids
    
    # Align columns - crucial for model prediction
    X_test = X_test[X.columns]
    
    print(f"\nComprehensive FE complete. Total features: {X.shape[1]}")
    gc.collect()
    
    return X, X_test, y_train


In [13]:
# =============================================================================
# BLOCK 2.5: EXECUTE FEATURE ENGINEERING
# =============================================================================
print("\n--- Starting Block 2.5: Executing Feature Engineering Pipeline ---")

# This is the crucial step that was missing.
# We call the function to create our training and testing dataframes.
X, X_test, y_train = create_comprehensive_features(df_train, df_test)

# Let's verify the output
print(f"Feature engineering complete. X shape: {X.shape}, X_test shape: {X_test.shape}")
gc.collect()


--- Starting Block 2.5: Executing Feature Engineering Pipeline ---
--- Starting Comprehensive Feature Engineering ---
Step 1: Creating brute-force numerical interaction features...
Step 2: Creating date features...
Step 3: Creating TF-IDF features for text columns...
Step 4: Creating group-by aggregation features...
Step 5: Creating ratio features...
Step 6: Creating geospatial clustering features...
Step 7: Finalizing feature set...

Comprehensive FE complete. Total features: 233
Feature engineering complete. X shape: (200000, 233), X_test shape: (200000, 233)


0

In [14]:
# =============================================================================
# BLOCK 3: PYTORCH SETUP & FULL DATA PREPARATION
# =============================================================================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import OneCycleLR # Using a more advanced scheduler
from sklearn.preprocessing import StandardScaler
import joblib

print(f"--- Starting Block 3: PyTorch Setup & Data Preparation ---")
print(f"PyTorch version: {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# --- Data Scaling (The Most Important Step for NNs) ---
# We scale the input features to have a mean of 0 and standard deviation of 1.
# This helps the network learn more efficiently.
print("\nScaling features for the Neural Network...")
feature_scaler = StandardScaler()
X_scaled = feature_scaler.fit_transform(X)
X_test_scaled = feature_scaler.transform(X_test)

# We ALSO scale the target variable. This stabilizes the training process for regression tasks.
print("Scaling target variable for the Neural Network...")
target_scaler = StandardScaler()
y_true_scaled = target_scaler.fit_transform(y_true.to_numpy().reshape(-1, 1))

# Save the scalers so we can inverse_transform the predictions later
joblib.dump(feature_scaler, 'feature_scaler.joblib')
joblib.dump(target_scaler, 'target_scaler.joblib')
print("\nFeature and target scalers created and saved.")
print("PyTorch setup and full data scaling complete.")

# --- Custom PyTorch Dataset (no changes needed here) ---
class HousePriceDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = features
        self.labels = labels
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        features = torch.tensor(self.features[idx], dtype=torch.float32)
        if self.labels is not None:
            labels = torch.tensor(self.labels[idx], dtype=torch.float32)
            return features, labels
        return features

--- Starting Block 3: PyTorch Setup & Data Preparation ---
PyTorch version: 2.7.1+cu126
Using device: cuda

Scaling features for the Neural Network...
Scaling target variable for the Neural Network...

Feature and target scalers created and saved.
PyTorch setup and full data scaling complete.


In [15]:
# =============================================================================
# BLOCK 4: DEFINE THE IMPROVED RESIDUAL NEURAL NETWORK ARCHITECTURE
# =============================================================================
print("\n--- Starting Block 4: Defining the Residual Neural Network ---")

class ResidualBlock(nn.Module):
    """A single residual block with Linear -> BatchNorm -> SiLU -> Dropout."""
    def __init__(self, input_size, output_size, dropout_rate):
        super(ResidualBlock, self).__init__()
        self.main_path = nn.Sequential(
            nn.Linear(input_size, output_size),
            nn.BatchNorm1d(output_size),
            nn.SiLU(),
            nn.Dropout(dropout_rate)
        )
        # If input and output sizes differ, we need a shortcut to match dimensions
        self.shortcut = nn.Identity() if input_size == output_size else nn.Linear(input_size, output_size)

    def forward(self, x):
        # The core idea: add the input (shortcut) to the output of the block
        return self.main_path(x) + self.shortcut(x)

class ResidualNet(nn.Module):
    """A full Neural Network built from a series of Residual Blocks."""
    def __init__(self, input_shape, layer_sizes, dropout_rates):
        super(ResidualNet, self).__init__()
        
        # An initial layer to process the input features
        layers = [nn.Linear(input_shape, layer_sizes[0]), nn.SiLU()]
        
        # Add the residual blocks
        in_size = layer_sizes[0]
        for i, (out_size, dropout) in enumerate(zip(layer_sizes, dropout_rates)):
            layers.append(ResidualBlock(in_size, out_size, dropout))
            in_size = out_size
            
        # Final output layer for regression
        layers.append(nn.Linear(in_size, 1))
        
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

print("Residual Neural Network architecture defined successfully.")


--- Starting Block 4: Defining the Residual Neural Network ---
Residual Neural Network architecture defined successfully.


In [15]:
# =============================================================================
# BLOCK 5: TUNE NN HYPERPARAMETERS WITH OPTUNA (REVISED)
# =============================================================================
import torch.optim as optim

# --- 1. Prepare a smaller dataset for faster tuning ---
print("--- Step 1: Preparing a smaller dataset for faster tuning... ---")
# We use a single 80/20 split of the SCALED data for this.
X_train_opt, X_val_opt, y_train_opt, y_val_opt = train_test_split(
    X_scaled, y_true_scaled, test_size=0.2, random_state=RANDOM_STATE
)

# Create DataLoaders for the tuning process
train_dataset_opt = HousePriceDataset(X_train_opt, y_train_opt)
val_dataset_opt = HousePriceDataset(X_val_opt, y_val_opt)
train_loader_opt = DataLoader(train_dataset_opt, batch_size=512, shuffle=True)
val_loader_opt = DataLoader(val_dataset_opt, batch_size=512, shuffle=False)
print("Tuning data prepared.")

# --- 2. Define the Optuna Objective Function for the Neural Network ---
def objective_nn(trial):
    """
    This function defines the search space and trains a NN for each Optuna trial.
    """
    # Define a search space for the key hyperparameters
    # We use suggest_categorical to choose between well-structured architectures.
    arch_choice = trial.suggest_categorical('architecture', ['arch_1', 'arch_2'])
    
    layer_sizes = {
        'arch_1': [512, 256, 128],
        'arch_2': [1024, 512, 256, 128]
    }[arch_choice]

    dropout_rates = {
        'arch_1': [trial.suggest_float(f'dr_1_{i}', 0.1, 0.5) for i in range(3)],
        'arch_2': [trial.suggest_float(f'dr_2_{i}', 0.1, 0.5) for i in range(4)]
    }[arch_choice]
    
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    
    # Initialize model, loss, and optimizer
    model = ResidualNet(
        input_shape=X_train_opt.shape[1],
        layer_sizes=layer_sizes,
        dropout_rates=dropout_rates
    ).to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    loss_fn = nn.HuberLoss()
    
    # Simplified training loop for one trial (fewer epochs, simple early stopping)
    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    for epoch in range(50): # Run for a max of 50 epochs per trial
        model.train()
        for features, labels in train_loader_opt:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(features)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for features, labels in val_loader_opt:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                val_loss += loss_fn(outputs, labels).item()
        
        current_val_loss = val_loss / len(val_loader_opt)
        
        if current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= 5: # Early stop after 5 epochs of no improvement
            break
            
    print(f"  Trial {trial.number}: Val Loss = {best_val_loss:.6f}")
    return best_val_loss

# --- 3. Create and Run the Optuna Study ---
study_nn = optuna.create_study(direction='minimize')
print("\n--- Starting Neural Network Hyperparameter Tuning... ---")
study_nn.optimize(objective_nn, n_trials=N_OPTUNA_TRIALS) # n_trials can be adjusted

# --- 4. Store the Best Parameters ---
# We store the final parameters in a clean dictionary.
best_params_nn = {
    'learning_rate': study_nn.best_params['learning_rate'],
    'weight_decay': study_nn.best_params['weight_decay'],
    'layer_sizes': {
        'arch_1': [512, 256, 128],
        'arch_2': [1024, 512, 256, 128]
    }[study_nn.best_params['architecture']],
    'dropout_rates': [study_nn.best_params[f'dr_{study_nn.best_params["architecture"][-1]}_{i}'] for i in range(len({
        'arch_1': [512, 256, 128],
        'arch_2': [1024, 512, 256, 128]
    }[study_nn.best_params['architecture']]))]
}
print("\n--- Neural Network Tuning Complete ---")
print(f"Best trial validation loss: {study_nn.best_value:.6f}")
print("Best hyperparameters found:")
print(best_params_nn)

--- Step 1: Preparing a smaller dataset for faster tuning... ---


[I 2025-07-22 18:09:31,509] A new study created in memory with name: no-name-29a772cb-f08d-4251-83dd-18aaef6311a6


Tuning data prepared.

--- Starting Neural Network Hyperparameter Tuning... ---


[I 2025-07-22 18:10:09,389] Trial 0 finished with value: 0.03677534375669835 and parameters: {'architecture': 'arch_1', 'dr_1_0': 0.2408108898355212, 'dr_1_1': 0.3459104609195603, 'dr_1_2': 0.3870451628594339, 'dr_2_0': 0.38477264526863664, 'dr_2_1': 0.2759180182015149, 'dr_2_2': 0.2349270200374036, 'dr_2_3': 0.10647275496375525, 'learning_rate': 0.0052710902280214385, 'weight_decay': 2.7469498535521866e-05}. Best is trial 0 with value: 0.03677534375669835.


  Trial 0: Val Loss = 0.036775


...

  Trial 27: Val Loss = 0.033731


[I 2025-07-22 18:40:48,143] Trial 28 finished with value: 0.033658449667729906 and parameters: {'architecture': 'arch_1', 'dr_1_0': 0.42490099741471016, 'dr_1_1': 0.18761196976479008, 'dr_1_2': 0.19026650813877863, 'dr_2_0': 0.2629599335625282, 'dr_2_1': 0.23068710931051262, 'dr_2_2': 0.10745479025688137, 'dr_2_3': 0.273851036125897, 'learning_rate': 0.0020890041020915273, 'weight_decay': 5.78551554223216e-06}. Best is trial 23 with value: 0.03220104768023461.


  Trial 28: Val Loss = 0.033658


[I 2025-07-22 18:41:52,035] Trial 29 finished with value: 0.03393162228167057 and parameters: {'architecture': 'arch_1', 'dr_1_0': 0.37159312980381365, 'dr_1_1': 0.15734833630406475, 'dr_1_2': 0.13405660718872847, 'dr_2_0': 0.1940837747641947, 'dr_2_1': 0.284861952703786, 'dr_2_2': 0.20207139923149287, 'dr_2_3': 0.3245824630456027, 'learning_rate': 0.0004867876083665849, 'weight_decay': 2.3636992572105354e-05}. Best is trial 23 with value: 0.03220104768023461.


  Trial 29: Val Loss = 0.033932

--- Neural Network Tuning Complete ---
Best trial validation loss: 0.032201
Best hyperparameters found:
{'learning_rate': 0.0007833213122687296, 'weight_decay': 2.2353912564761073e-06, 'layer_sizes': [512, 256, 128], 'dropout_rates': [0.38831206715780076, 0.1380700478405189, 0.11587246202254503]}


In [16]:
# =============================================================================
# BLOCK 5.5: DEFINE BEST HYPERPARAMETERS MANUALLY
# =============================================================================
import torch.optim as optim

# --- 1. Prepare a smaller dataset for faster tuning ---
print("--- Step 1: Preparing a smaller dataset for faster tuning... ---")
# We use a single 80/20 split of the SCALED data for this.
X_train_opt, X_val_opt, y_train_opt, y_val_opt = train_test_split(
    X_scaled, y_true_scaled, test_size=0.2, random_state=RANDOM_STATE
)

# Create DataLoaders for the tuning process
train_dataset_opt = HousePriceDataset(X_train_opt, y_train_opt)
val_dataset_opt = HousePriceDataset(X_val_opt, y_val_opt)
train_loader_opt = DataLoader(train_dataset_opt, batch_size=512, shuffle=True)
val_loader_opt = DataLoader(val_dataset_opt, batch_size=512, shuffle=False)
print("Tuning data prepared.")
print("--- Defining the best hyperparameters from the previous Optuna run ---")

# These are the exact results from your successful tuning process (Trial 23).
best_params_nn = {
    'learning_rate': 0.0007833213122687296,
    'weight_decay': 2.2353912564761073e-06,
    'layer_sizes': [512, 256, 128],
    'dropout_rates': [
        0.38831206715780076,
        0.1380700478405189,
        0.11587246202254503
    ]
}

print("Successfully created the 'best_params_nn' dictionary:")
print(best_params_nn)

--- Step 1: Preparing a smaller dataset for faster tuning... ---
Tuning data prepared.
--- Defining the best hyperparameters from the previous Optuna run ---
Successfully created the 'best_params_nn' dictionary:
{'learning_rate': 0.0007833213122687296, 'weight_decay': 2.2353912564761073e-06, 'layer_sizes': [512, 256, 128], 'dropout_rates': [0.38831206715780076, 0.1380700478405189, 0.11587246202254503]}


In [19]:
# =============================================================================
# BLOCK 6 & 7 (FORENSIC): K-FOLD TRAINING, DEEP ANALYSIS & SAVING
# =============================================================================
import torch
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import OneCycleLR
import gc

# --- 1. Training Configuration ---
print("--- Step 1: Loading Training Configuration ---")
print("Using the following optimal hyperparameters found by Optuna:")
print(best_params_nn)
EPOCHS = 200
BATCH_SIZE = 512
PATIENCE = 20

# --- 2. Initialize Prediction Arrays & K-Fold ---
print("\n--- Step 2: Initializing Prediction Arrays and K-Fold Strategy ---")
oof_nn_preds = np.zeros(len(X))
test_nn_preds = np.zeros(len(X_test)) # This will accumulate summed predictions
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
print(f"Initialized {N_SPLITS}-Fold Stratified Cross-Validation.")


for fold, (train_idx, val_idx) in enumerate(skf.split(X_scaled, grade_for_stratify)):
    
    # --- A. Setup and Training for the current fold ---
    print("\n" + "="*80)
    print(f"--- FORENSIC ANALYSIS: TRAINING FOLD {fold+1}/{N_SPLITS} ---")
    print("="*80)

    # (Setup and training loop is correct and included for completeness)
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y_true_scaled[train_idx], y_true_scaled[val_idx]
    train_dataset = HousePriceDataset(X_train, y_train)
    val_dataset = HousePriceDataset(X_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    print(f"Fold {fold+1} data prepared. Train size: {len(X_train)}, Val size: {len(X_val)}")
    model = ResidualNet(input_shape=X_train.shape[1], layer_sizes=best_params_nn['layer_sizes'], dropout_rates=best_params_nn['dropout_rates']).to(device)
    loss_fn = nn.HuberLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=best_params_nn['learning_rate'], weight_decay=best_params_nn['weight_decay'])
    scheduler = OneCycleLR(optimizer, max_lr=best_params_nn['learning_rate'], epochs=EPOCHS, steps_per_epoch=len(train_loader))
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_state = None
    for epoch in range(EPOCHS):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(features)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for features, labels in val_loader:
                val_loss += loss_fn(model(features.to(device)), labels.to(device)).item()
        val_loss /= len(val_loader)
        if (epoch + 1) % 10 == 0: print(f"  Epoch {epoch+1:03d} | Validation Loss: {val_loss:.6f}")
        if val_loss < best_val_loss:
            best_val_loss, epochs_no_improve, best_model_state = val_loss, 0, model.state_dict().copy()
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}. Best validation loss: {best_val_loss:.6f}")
            break

    # --- C. Generate and IMMEDIATELY INSPECT Predictions ---
    print(f"\nFold {fold+1} training complete. Generating and inspecting predictions...")
    model.load_state_dict(best_model_state)
    model.eval()

    # --- OOF predictions ---
    val_preds_list = []
    with torch.no_grad():
        for features, _ in val_loader:
            outputs = model(features.to(device))
            # === THE FIX IS HERE ===
            # We must .detach() the tensor from the computation graph before converting to numpy
            val_preds_list.append(outputs.detach().cpu().numpy())
    raw_oof_preds = np.concatenate(val_preds_list)
    oof_nn_preds[val_idx] = target_scaler.inverse_transform(raw_oof_preds).flatten()

    # --- FORENSIC INSPECTION OF TEST PREDICTIONS ---
    test_preds_list = []
    test_loader_fold = DataLoader(HousePriceDataset(X_test_scaled), batch_size=BATCH_SIZE*2, shuffle=False)
    with torch.no_grad():
        for features in test_loader_fold:
            outputs = model(features.to(device))
            # === THE FIX IS HERE (AGAIN) ===
            test_preds_list.append(outputs.detach().cpu().numpy())
    raw_test_preds = np.concatenate(test_preds_list)
    
    # (The forensic analysis part remains the same)
    print("\n" + "-"*25 + f" ANALYSIS FOR FOLD {fold+1} " + "-"*25)
    print("Describing RAW (SCALED) test predictions (direct model output):")
    print(pd.Series(raw_test_preds.flatten()).describe())
    inversed_test_preds = target_scaler.inverse_transform(raw_test_preds).flatten()
    print("\nDescribing INVERSE-TRANSFORMED test predictions for this fold:")
    print(pd.Series(inversed_test_preds).describe())
    if np.min(inversed_test_preds) < -10000:
        print(f"\n*** BUG DETECTED IN FOLD {fold+1}: Extreme negative values found. ***")
    else:
        print(f"\n  Fold {fold+1} predictions appear clean.")
    print("-"*65)
        
    test_nn_preds += inversed_test_preds
    del model, X_train, X_val, y_train, y_val, train_loader, val_loader, test_loader_fold, raw_test_preds, inversed_test_preds
    gc.collect()
# --- 4. Finalize, Evaluate, and Conduct Final Analysis ---
print("\n" + "="*80)
print("--- K-Fold Training Complete: Final Analysis and Saving ---")
print("="*80)

test_nn_preds /= N_SPLITS
print("\n--- Step 4: Finalizing Predictions and Evaluating OOF Score ---")
final_mean_rmse_nn = np.sqrt(mean_squared_error(y_true, oof_nn_preds))
print(f"Final NN Mean Model OOF RMSE: ${final_mean_rmse_nn:,.2f}")

print("\n--- Step 4.5: Final Forensic Analysis of FINAL AVERAGED Prediction Arrays ---")
print("\nDescribing final 'oof_nn_preds':")
print(pd.Series(oof_nn_preds).describe())
print("\nDescribing final 'test_nn_preds':")
print(pd.Series(test_nn_preds).describe())
if np.isnan(test_nn_preds).any() or np.min(test_nn_preds) < 0:
    print("\n*** CRITICAL WARNING: Final averaged test predictions are still corrupt! ***")
else:
    print("\nSUCCESS: Final averaged test predictions appear clean and reasonable.")

# --- 5. Save the Prediction Arrays ---
print("\n--- Step 5: Saving Prediction Arrays ---")
SAVE_PATH = './NN_model_predictions/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"Prediction arrays will be saved in: {SAVE_PATH}")
np.save(os.path.join(SAVE_PATH, 'oof_nn_preds.npy'), oof_nn_preds)
np.save(os.path.join(SAVE_PATH, 'test_nn_preds.npy'), test_nn_preds)
print("\nOOF and Test prediction arrays saved successfully.")

--- Step 1: Loading Training Configuration ---
Using the following optimal hyperparameters found by Optuna:
{'learning_rate': 0.0007833213122687296, 'weight_decay': 2.2353912564761073e-06, 'layer_sizes': [512, 256, 128], 'dropout_rates': [0.38831206715780076, 0.1380700478405189, 0.11587246202254503]}

--- Step 2: Initializing Prediction Arrays and K-Fold Strategy ---
Initialized 5-Fold Stratified Cross-Validation.

--- FORENSIC ANALYSIS: TRAINING FOLD 1/5 ---
Fold 1 data prepared. Train size: 160000, Val size: 40000
  Epoch 010 | Validation Loss: 0.045018
  Epoch 020 | Validation Loss: 0.039532
  Epoch 030 | Validation Loss: 0.037084
  Epoch 040 | Validation Loss: 0.036287
  Epoch 050 | Validation Loss: 0.034023
  Epoch 060 | Validation Loss: 0.035003
  Epoch 070 | Validation Loss: 0.033369
  Epoch 080 | Validation Loss: 0.033092
  Epoch 090 | Validation Loss: 0.033265

Early stopping at epoch 98. Best validation loss: 0.032185

Fold 1 training complete. Generating and inspecting predi

KeyboardInterrupt: 

In [25]:
# =============================================================================
# BLOCK: FINAL FORENSIC ANALYSIS OF NN PREDICTIONS
# =============================================================================
print("\n" + "="*60)
print("--- Step 4.5: Final Forensic Analysis of Prediction Arrays ---")
print("="*60)
print("This step provides a statistical summary of the final prediction arrays to ensure they are not corrupted.")

# --- Analyze the Out-of-Fold (OOF) Predictions ---
# This is our "ground truth" for what good predictions should look like.
print("\nDescribing 'oof_nn_preds' (predictions on the training set):")
oof_series = pd.Series(oof_nn_preds)
print(oof_series.describe())

# --- Analyze the Test Set Predictions ---
# This is the array that was previously corrupted. We need to verify it's now fixed.
print("\nDescribing 'test_nn_preds' (final predictions on the test set):")
test_series = pd.Series(test_nn_preds)
print(test_series.describe())

# --- Automated Integrity Check ---
# This check programmatically looks for any NaN or infinite values.
print("\n--- Automated Integrity Check ---")
oof_clean = not (np.isnan(oof_nn_preds).any() or np.isinf(oof_nn_preds).any())
test_clean = not (np.isnan(test_nn_preds).any() or np.isinf(test_nn_preds).any())

if oof_clean and test_clean:
    print("SUCCESS: Both OOF and Test prediction arrays are clean (no NaN or Inf values).")
    print("The statistics for the test set predictions appear reasonable and in line with the OOF predictions.")
    print("You are now clear to proceed with saving the files.")
else:
    print("\n*** CRITICAL WARNING: Corrupted values (NaN or Inf) detected! Do NOT proceed. ***")
    if not oof_clean:
        print("*** The 'oof_nn_preds' array is corrupted. ***")
    if not test_clean:
        print("*** The 'test_nn_preds' array is corrupted. ***")


--- Step 4.5: Final Forensic Analysis of Prediction Arrays ---
This step provides a statistical summary of the final prediction arrays to ensure they are not corrupted.

Describing 'oof_nn_preds' (predictions on the training set):
count    2.000000e+05
mean     5.799302e+05
std      4.052718e+05
min     -1.742588e+04
25%      3.073598e+05
50%      4.606330e+05
75%      7.147878e+05
max      3.630945e+06
dtype: float64

Describing 'test_nn_preds' (final predictions on the test set):
count    2.000000e+05
mean    -1.488278e+08
std      6.682088e+10
min     -2.988321e+13
25%      3.108083e+05
50%      4.678495e+05
75%      7.255352e+05
max      3.401911e+06
dtype: float64

--- Automated Integrity Check ---
SUCCESS: Both OOF and Test prediction arrays are clean (no NaN or Inf values).
The statistics for the test set predictions appear reasonable and in line with the OOF predictions.
You are now clear to proceed with saving the files.


In [26]:
# --- 5. Save the Prediction Arrays ---
print("\n--- Step 5: Saving Prediction Arrays ---")
SAVE_PATH = './NN_model_predictions/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"Prediction arrays will be saved in: {SAVE_PATH}")

try:
    # Save the OOF predictions (for error modeling) and the Test predictions (for submission)
    np.save(os.path.join(SAVE_PATH, 'oof_nn_preds.npy'), oof_nn_preds)
    np.save(os.path.join(SAVE_PATH, 'test_nn_preds.npy'), test_nn_preds)
    print("\nOOF and Test prediction arrays saved successfully.")
    print("You can now load these .npy files in your final ensembling notebook.")
except Exception as e:
    print(f"\nAn error occurred while saving the files: {e}")


--- Step 5: Saving Prediction Arrays ---
Prediction arrays will be saved in: ./NN_model_predictions/

OOF and Test prediction arrays saved successfully.
You can now load these .npy files in your final ensembling notebook.
